In [ ]:
import pysvorbit
import matplotlib.pyplot as plt
import numpy as np
from backend import _solve_kepler

In [ ]:
target = "GJ 586Aab (HD 137763, WDS 15282-0921)"

In [ ]:
num_points = 10000
# Sky projected orbit plotter function
def get_sky_projected_orbit(a, e, i, omega, Omega, num_points):

    # Computes sky-projected coordinates (X, Y) from Keplerian elements. Angles (i, omega, Omega) should be in degrees.
    
    # Convert angles to radians
    i_rad = np.radians(i)
    omega_rad = np.radians(omega)
    Omega_rad = np.radians(Omega)
    
    # 1. Sample true anomalies across a full orbit
    nu = np.linspace(0, 2 * np.pi, num_points)
    
    # 2. Distance 'r' from the focus for an ellipse
    r = (a * (1 - e**2)) / (1 + e * np.cos(nu))
    
    # Position vectors in the intrinsic orbital plane
    x_prime = r * np.cos(nu)
    y_prime = r * np.sin(nu)
    
    # 3. Apply Thiele-Innes or standard rotation matrix combinations
    # X (East) and Y (North) positions projected on the plane of the sky
    X = x_prime * (np.cos(omega_rad) * np.sin(Omega_rad) + np.sin(omega_rad) * np.cos(Omega_rad) * np.cos(i_rad)) + \
        y_prime * (-np.sin(omega_rad) * np.sin(Omega_rad) + np.cos(omega_rad) * np.cos(Omega_rad) * np.cos(i_rad))
        
    Y = x_prime * (np.cos(omega_rad) * np.cos(Omega_rad) - np.sin(omega_rad) * np.sin(Omega_rad) * np.cos(i_rad)) + \
        y_prime * (-np.sin(omega_rad) * np.cos(Omega_rad) - np.cos(omega_rad) * np.sin(Omega_rad) * np.cos(i_rad))
        
    return X, Y



def sky_position_at_epoch(t, P, T, e, a, i, omega, Omega):
    """
    Model position on the sky at epoch(s) t, using the same projection as
    get_sky_projected_orbit(). 'a' in the units you want back (mas here);
    angles in degrees; t, T, P all in years.
    Returns (North, East) so it plots directly on your (x = North, y = East) axes.
    """
    t = np.atleast_1d(np.asarray(t, dtype=float))
 
    # Mean anomaly at each epoch
    M = 2 * np.pi * (((t - T) / P) % 1.0)
 
    # Eccentric anomaly from PySVOrbit's own Kepler solver (one epoch at a time)
    E = np.array([_solve_kepler(m, e) for m in M])
 
    # Safety check: _solve_kepler can occasionally diverge at very high e
    bad = np.abs(E - e * np.sin(E) - M) > 1e-6
    if bad.any():
        raise RuntimeError(f"_solve_kepler did not converge for epoch(s) {t[bad]}")
 
    # True anomaly and orbital radius
    nu = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    r = a * (1 - e**2) / (1 + e * np.cos(nu))
    xp, yp = r * np.cos(nu), r * np.sin(nu)
 
    io, wo, Oo = np.radians([i, omega, Omega])
    East = xp * (np.cos(wo) * np.sin(Oo) + np.sin(wo) * np.cos(Oo) * np.cos(io)) + \
           yp * (-np.sin(wo) * np.sin(Oo) + np.cos(wo) * np.cos(Oo) * np.cos(io))
    North = xp * (np.cos(wo) * np.cos(Oo) - np.sin(wo) * np.sin(Oo) * np.cos(io)) + \
            yp * (-np.sin(wo) * np.cos(Oo) - np.cos(wo) * np.sin(Oo) * np.cos(io))
    return North, East


In [ ]:
P_2016 = 2.43623
T_2016 = 2007.2603
a_2016 = 0.1074*1000      # Semi-major axis mas
e_2016 = 0.976    # Eccentricity
i_2016 = 55.4     # Inclination (degrees)
omega_2016 = 255.6  # Argument of periapsis (degrees)
Omega_2016 = 272.8  # Longitude of ascending node (degrees)
plx_2016 = 52.5 # +/- 6.2 mas
X_2016, Y_2016 = get_sky_projected_orbit(a_2016, e_2016, i_2016, omega_2016, Omega_2016, num_points)

In [ ]:
# Load and fit SOAR + MIRC + Strass RV data

# Load the MIRC + SOAR + Strass RV data 
data_path = "SOAR + MIRC + Strass2013RV/GJ586A_SOAR_MIRC_StrassRV.inp"
pysvorbit.readinp(data_path)

# Uses scipy least_squares() to fit the orbit
pysvorbit.fitorb() 

pysvorbit.orbsave()   # writes <name>_output.csv to the working directory

pysvorbit.orbplot(ps=True) # plots fitted orbit and closes figs
pysvorbit.residual_plots()




In [ ]:
# Values from the PySVOrbit fit on GJ586A_visual.inp
P, T, e, a, W, w, i, K1, K2, V0 = pysvorbit.orb.el
P_err, T_err, e_err, a_err, W_err, w_err, i_err, K1_err, K2_err, V0_err = pysvorbit.orb.elerr
# Plot the recent PySVOrbit fit
X_test, Y_test = get_sky_projected_orbit(a*1e3, e, i, w, W, num_points)

print("PySVOrbit Fit Results:\n")
for name, value, error in zip(pysvorbit.orb.elname,
                              pysvorbit.orb.el,
                              pysvorbit.orb.elerr):
    print(f"{name} = {value} ± {error}")

In [ ]:
plt.close('all')

In [ ]:
# PMOIRED chi2r Color Map Data 

# Taking any chi2r under 4
data1 = np.genfromtxt("PMOIRED_chi2r/2014_07_29_chi2r_sep_PA.csv", delimiter=",", skip_header=1, usecols = (0, 1, 2))

chi2r_1 = data1[:, 0]  
sep_1 = data1[:, 1]                
PA_deg_1 = data1[:, 2]   

PA_1 = np.deg2rad(PA_deg_1)

x_obs_1 = sep_1 * np.cos(PA_1)
y_obs_1 = sep_1 * np.sin(PA_1)


# Taking the first 1000 fits
data2 = np.genfromtxt("PMOIRED_chi2r/2014_07_26_chi2r_sep_PA.csv", delimiter=",", skip_header=1, usecols = (0, 1, 2))

chi2r_2 = data2[:, 0]  
sep_2 = data2[:, 1]                
PA_deg_2 = data2[:, 2]   

PA_2 = np.deg2rad(PA_deg_2)

x_obs_2 = sep_2 * np.cos(PA_2)
y_obs_2 = sep_2 * np.sin(PA_2)


# Min chi2r of 2014-7-29
min_chi2r_1 = np.min(chi2r_1)
sep_min_chi2r_1 = sep_1[np.argmin(chi2r_1)]
PA_min_chi2r_1 = PA_1[np.argmin(chi2r_1)]

# Min chi2r of 2014-7-26
min_chi2r_2 = np.min(chi2r_2)
sep_min_chi2r_2 = sep_2[np.argmin(chi2r_2)]
PA_min_chi2r_2 = PA_2[np.argmin(chi2r_2)]

In [ ]:
# PMOIRED Chi2r Plots
fig, ax = plt.subplots(figsize=(12, 8))

"""
# Color map of chi2r for 2014-7-26
tcf2 = ax.tricontourf(y_obs_2, x_obs_2, chi2r_2,
                     levels=15, cmap='viridis_r', alpha=0.8, zorder=1)
ax.tricontour(y_obs_2, x_obs_2, chi2r_2, levels=15,
              colors='k', linewidths=0.3, zorder=1)
fig.colorbar(tcf2, ax=ax, shrink=0.8).set_label(r"Reduced $\chi^2$")


# Plot 2014-7-26 TOkovinin 2016 Prediction
x_7_26, y_7_26 = sky_position_at_epoch(2014.565024, P_2016, T_2016, e_2016, a_2016, i_2016, omega_2016, Omega_2016)
ax.scatter(y_7_26, x_7_26, marker='o', color='cyan', label='2014-07-26 Tokovinin 2016 Prediction', zorder = 10)

# Plot PMOIRED fits for 2014-7-26
ax.scatter(y_obs_2, x_obs_2, marker = "x", color="blue", label="2014-07-26 PMOIRED OIFITS (Top 459 Fits given by PMOIRED)", zorder=11)

# Plot the best chi2r for 2014-7-26
ax.scatter(sep_min_chi2r_2*np.sin(PA_min_chi2r_2), sep_min_chi2r_2*np.cos(PA_min_chi2r_2), marker='2', color='black', label='2014-07-26 PMOIRED Lowest Chi2r')
"""


"""

# Color map of chi2r for 2014-7-29
tcf = ax.tricontourf(y_obs_1, x_obs_1, chi2r_1,
                     levels=15, cmap='viridis_r', alpha=0.8, zorder=1)
ax.tricontour(y_obs_1, x_obs_1, chi2r_1, levels=15,
              colors='k', linewidths=0.3, zorder=1)
fig.colorbar(tcf, ax=ax, shrink=0.8).set_label(r"Reduced $\chi^2$")
"""

# Plot 2014-7-29 TOkovinin 2016 Prediction
x_7_29, y_7_29 = sky_position_at_epoch(2014.573105, P_2016, T_2016, e_2016, a_2016, i_2016, omega_2016, Omega_2016)
ax.scatter(y_7_29, x_7_29, marker='o', color='cyan', label='2014-07-29 Tokovinin 2016 Prediction')


# Plot PMOIRED fits for 2014-7-29
ax.scatter(y_obs_1, x_obs_1, marker = "x", color="red", label="2014-07-29 PMOIRED OIFITS (chi2r < 4, n = 316)")

# Plot the best chi2r for 2014-7-29
ax.scatter(sep_min_chi2r_1*np.sin(PA_min_chi2r_1), sep_min_chi2r_1*np.cos(PA_min_chi2r_1), marker='2', color='black', label='2014-07-29 PMOIRED Lowest Chi2r')




# Zoom window. Change the limits and rerun only this cell.
centerx = -7.8
centery = 1
half_width = 10


"""
# --------------------------------------_
# Check for the PMOIRED 2014-7-29 Fits that are on the Monte-Carlo Orbits and Tokovinin 2016
# Keep 2014-07-29 rows with separation in [7, 9] mas and position angle in [260, 280] deg.
keep = (sep_1 >= 7) & (sep_1 <= 9) & (PA_deg_1 >= 265) & (PA_deg_1 <= 280)
chi2r_filt = chi2r_1[keep].tolist()
sep_filt = sep_1[keep].tolist()
PA_filt = PA_deg_1[keep].tolist()


# Same sky convention as the other PMOIRED points: East = sep*sin(PA), North = sep*cos(PA).
PA_filt_rad = np.deg2rad(PA_filt)
north_filt = np.array(sep_filt) * np.cos(PA_filt_rad)
east_filt = np.array(sep_filt) * np.sin(PA_filt_rad)
ax.scatter(east_filt, north_filt, marker="x", color="purple", label="2014-07-29 filtered")
# --------------------------------------_
"""


ax.plot(X_test, Y_test, color="red", lw=1.2, zorder=3, label="PySVOrbit Best Fit")
ax.plot(X_2016, Y_2016, color="blue", linestyle="-", lw=0.5, zorder=2, label="Tokovinin 2016 Fit")
ax.add_patch(plt.Circle((0, 0), 1, facecolor="orange"))
ax.set_aspect("equal", adjustable="datalim")
ax.invert_xaxis()  # East to the left, so the limits are written right-to-left

ax.set_xlim(centerx + half_width, centerx - half_width)
ax.set_ylim(centery - half_width, centery + half_width)

ax.grid(alpha=0.3)
ax.set_xlabel(r"$\leftarrow$ E (mas)")
ax.set_ylabel(r"$\rightarrow$ N (mas)")
ax.set_title(f"{target}")
ax.legend(fontsize=8, loc="upper left")
plt.show()


In [ ]:
# --------------------------------------_
# Check for the PMOIRED 2014-7-29 Fits that are on the Monte-Carlo Orbits and Tokovinin 2016
# Keep 2014-07-29 rows with separation in [7, 9] mas and position angle in [260, 280] deg.
keep = (sep_1 >= 7) & (sep_1 <= 9) & (PA_deg_1 >= 265) & (PA_deg_1 <= 280)
chi2r_filt = chi2r_1[keep].tolist()
sep_filt = sep_1[keep].tolist()
PA_filt = PA_deg_1[keep].tolist()



In [ ]:
print(np.median(sep_filt))
print(np.median(PA_filt))

print(f'2014-07-29 Separation\n  {np.median(sep_filt)} ± {np.std(sep_filt, ddof=1)/np.sqrt(len(sep_filt))} mas')
print(f'2014-07-29 Position Angle\n  {np.median(PA_filt)} deg')
